<a href="https://colab.research.google.com/github/jupinzonri/Taller-5/blob/main/Taller_5_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller 5: Introducción a las Redes Neuronales Artificiales
**Nombre:** Juan Felipe Pinzón Rincón

## Análisis de los Ejercicios Seleccionados

### 1. Predicción con base en datos experimentales (Regresión con RNA)
* Se utiliza un conjunto de datos experimentales simulados. En este escenario, la red neuronal artificial procesa entradas continuas para generar una predicción numérica (regresión).
* La topología de la red incluye dos capas ocultas. Como se menciona en el documento, la función de activación ReLU es ideal en las capas internas de redes profundas porque mejora la eficiencia del entrenamiento.
* Se dividen los datos en entrenamiento y prueba, aplicando el descenso del gradiente para actualizar gradualmente los pesos a través del algoritmo de retropropagación (backpropagation), reduciendo el error cuadrático medio.

### 2. Clasificación de prendas con Fashion MNIST (Evaluación de Precisión)
* El dataset Fashion MNIST consta de imágenes en escala de grises de $28\times28$ píxeles, similar al MNIST original de dígitos, y busca clasificar cada imagen en una de 10 categorías de prendas de vestir.
* La red se construye con una capa inicial que "aplana" (Flatten) la matriz de $28\times28$ en un vector de 784 características, seguida de dos capas ocultas densas con activación ReLU.
* La capa de salida consta de 10 neuronas con activación `softmax`, la cual produce probabilidades entre 0 y 1 para cada clase. Al final se evalúa la precisión (accuracy) del modelo sobre el conjunto de prueba para validar que no hubo sobre-entrenamiento (overfitting).

### 3. Explicación de funciones principales en Fashion MNIST
* **`Flatten`**: Recibe la imagen bidimensional de los datos de entrada ($28\times28$) y la convierte en un vector unidimensional de 784 píxeles, preparándola para la red neuronal densa.
* **`Dense`**: Representa una capa de neuronas completamente conectadas. Los pesos $W$ y los sesgos $b$ de esta capa se ajustan iterativamente durante el aprendizaje para minimizar la función de pérdida.
* **`ReLU` (Rectified Linear Unit)**: Función de activación $f(x) = \max(0, x)$ que introduce no linealidad y complejidad a la red, resolviendo ineficiencias del entrenamiento presentes en funciones como la sigmoide.
* **`Softmax`**: Función aplicada en la capa de salida que asegura que la suma de las predicciones de las 10 clases sea igual a 1, entregando una distribución de probabilidad clara para la clasificación final.

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# =====================================================================
# EJERCICIO 1: Modelo de predicción con datos experimentales
# =====================================================================
def ejercicio_1_prediccion_experimental():
    print("--- EJERCICIO 1: PREDICCIÓN CON DATOS EXPERIMENTALES ---")

    # Simulación de un dataset experimental (Ej: Voltaje vs Temperatura)
    np.random.seed(42)
    X_exp = np.linspace(0, 10, 200).reshape(-1, 1)
    # Función no lineal con ruido
    Y_exp = 3 * np.sin(X_exp) + 0.5 * X_exp + np.random.randn(200, 1) * 0.5

    # Topología: 2 capas ocultas
    modelo_exp = keras.Sequential([
        layers.Dense(16, activation='relu', input_shape=[1]), # Capa Oculta 1
        layers.Dense(16, activation='relu'),                  # Capa Oculta 2
        layers.Dense(1)                                       # Capa de Salida (Lineal)
    ])

    # Compilar con error cuadrático medio (MSE)
    modelo_exp.compile(optimizer='adam', loss='mean_squared_error')

    # Entrenamiento
    print("Entrenando modelo experimental...")
    historial = modelo_exp.fit(X_exp, Y_exp, epochs=150, verbose=0, validation_split=0.2)

    # Predicción con nuevos datos
    X_nuevo = np.array([[2.5], [5.0], [7.5]])
    predicciones = modelo_exp.predict(X_nuevo)

    print("Predicciones para nuevos datos:")
    for x_val, y_pred in zip(X_nuevo, predicciones):
        print(f"Entrada X: {x_val[0]:.2f} -> Predicción Y: {y_pred[0]:.2f}")

    print(f"Pérdida final (MSE): {historial.history['loss'][-1]:.4f}\n")


# =====================================================================
# EJERCICIO 2 y 3: Clasificación Fashion MNIST con 2 Capas Ocultas
# =====================================================================
def ejercicios_2_y_3_fashion_mnist():
    print("--- EJERCICIOS 2 Y 3: FASHION MNIST ---")

    # Descargar el dataset Fashion MNIST
    fashion_mnist = keras.datasets.fashion_mnist
    (train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

    # Nombres de las clases para referencia
    class_names = ['Camiseta', 'Pantalón', 'Saco', 'Vestido', 'Abrigo',
                   'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Bota']

    # Normalización: Escalar valores de pixeles entre 0 y 1
    train_images = train_images / 255.0
    test_images = test_images / 255.0

    # Topología de la Red: 2 capas ocultas
    modelo_fashion = keras.Sequential([
        layers.Flatten(input_shape=(28, 28)), # Capa de Entrada: Aplana matriz 28x28 a vector 784
        layers.Dense(128, activation='relu'), # Capa Oculta 1
        layers.Dense(64, activation='relu'),  # Capa Oculta 2
        layers.Dense(10, activation='softmax')# Capa de Salida: 10 clases, probabilidad
    ])

    # Compilación: Función de pérdida para categorías (rótulos enteros)
    modelo_fashion.compile(optimizer='adam',
                           loss='sparse_categorical_crossentropy',
                           metrics=['accuracy'])

    # Entrenamiento
    print("Entrenando modelo Fashion MNIST...")
    historial_fashion = modelo_fashion.fit(train_images, train_labels, epochs=10, validation_split=0.1, verbose=1)

    # Evaluación con el conjunto de prueba
    test_loss, test_acc = modelo_fashion.evaluate(test_images,  test_labels, verbose=2)
    print(f"\nPrecisión en el conjunto de prueba: {test_acc:.4f}")

    # Conclusiones impresas
    print("\n--- Conclusiones ---")
    print("1. La precisión en el conjunto de prueba indica qué tan bien la red neuronal generaliza patrones de ropa (detectores locales) a datos que nunca ha visto.")
    print("2. Si la precisión de entrenamiento es mucho mayor que la de prueba, tendríamos un problema de 'Overfitting' (sobre-entrenamiento).")
    print("3. La función 'softmax' fue clave porque permite que la red entregue una probabilidad definida (Ej: 90% Bota, 10% Sandalia) facilitando la clasificación final.")


# Ejecución de los scripts
if __name__ == "__main__":
    ejercicio_1_prediccion_experimental()
    ejercicios_2_y_3_fashion_mnist()

--- EJERCICIO 1: PREDICCIÓN CON DATOS EXPERIMENTALES ---
Entrenando modelo experimental...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
Predicciones para nuevos datos:
Entrada X: 2.50 -> Predicción Y: 1.77
Entrada X: 5.00 -> Predicción Y: 2.72
Entrada X: 7.50 -> Predicción Y: 3.67
Pérdida final (MSE): 4.3444

--- EJERCICIOS 2 Y 3: FASHION MNIST ---
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Entrenando modelo Fashion MNIST...
Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8217 - loss: 0.5020 - val_accuracy: 0.8443 - val_loss: 0.4274
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8624 - loss: 0.3770 - val_accuracy: 0.8658 - val_loss: 0.3651
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.8746 - loss: 0.3401 - val_accuracy: 0.8645 - val_loss: 0.3752
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8842 - loss: 0.3156 - val_accuracy: 0.8730 - val_loss: 0.3395
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8897 - loss: 0.2981 - val_accuracy: 0.8803 - val_loss: 0.3273
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8953 - loss: 0.2825 - val_accuracy: 0.8847 - val_loss: 0.3310
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8999 - loss: 0.2687 - val_accuracy: 0.8865 - val_loss: 0.3150
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step 